# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hapepaAhmed/my-capstone-project/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip install -q huggingface_hub

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import userdata
from huggingface_hub import hf_hub_download

In [3]:
HF_TOKEN = userdata.get("HF_TOKEN")

In [4]:
parquet_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

In [5]:
df = pd.read_parquet(parquet_path)

print(df.shape)
df.head()

(9841378, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

The baseline prioritizes content pages that already receive substantial search visibility but do not rank near the top of search results. These pages represent promising optimization opportunities because improving their rankings may increase organic traffic.

### Signals Used

- Google Search impressions (`gsc_impressions`)
- Average search position (`gsc_avg_position`)

### Reason Codes

| Reason Code | Meaning |
|--------------|---------|
| HIGH_IMPRESSIONS_LOW_RANK | High visibility but poor ranking; prioritize optimization. |
| LOW_IMPRESSIONS | Limited search visibility; monitor rather than optimize immediately. |
| GOOD_POSITION | Already ranks well; maintain current performance. |
| REVIEW | Mixed signals requiring manual review. |

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# Make a copy
baseline = df.copy()

# Normalize the signals (0-1 scale)
baseline["impressions_norm"] = (
    baseline["gsc_impressions"] - baseline["gsc_impressions"].min()
) / (
    baseline["gsc_impressions"].max() - baseline["gsc_impressions"].min()
)

baseline["position_norm"] = (
    baseline["gsc_avg_position"] - baseline["gsc_avg_position"].min()
) / (
    baseline["gsc_avg_position"].max() - baseline["gsc_avg_position"].min()
)

# Higher impressions + worse position = higher priority
baseline["baseline_score"] = (
    baseline["impressions_norm"] +
    baseline["position_norm"]
) / 2

# Reason code
high_impressions = (
    baseline["gsc_impressions"] >=
    baseline["gsc_impressions"].median()
)

good_position = (
    baseline["gsc_avg_position"] <=
    baseline["gsc_avg_position"].median()
)

poor_position = ~good_position

# Action label
conditions = [

    high_impressions & poor_position,

    ~high_impressions,

    good_position

]

choices = [

    "HIGH_IMPRESSIONS_LOW_RANK",

    "LOW_IMPRESSIONS",

    "GOOD_POSITION"

]

baseline["reason_code"] = np.select(
    conditions,
    choices,
    default="REVIEW"
)

action_map = {

    "HIGH_IMPRESSIONS_LOW_RANK":
        "Optimize Immediately",

    "LOW_IMPRESSIONS":
        "Monitor",

    "GOOD_POSITION":
        "Protect Performance",

    "REVIEW":
        "Manual Review"

}

baseline["action"] = (
    baseline["reason_code"]
    .map(action_map)
)
# Rank pages
baseline = baseline.sort_values(
    by="baseline_score",
    ascending=False
)

baseline.head(10)



,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,impressions_norm,position_norm,baseline_score,reason_code,action
9655081,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,True,True,True,False,40084,1,3341,...,0.0,0.0,0.0,0.0,0.0,1.000000,0.000167,0.500084,GOOD_POSITION,Protect Performance
3780826,2026-03-14,client_08a6a72ff48e62c0,content_8390ee56e6ea98ee,True,False,True,None,1,0,498,...,NaN,NaN,NaN,NaN,NaN,0.000025,1.000000,0.500012,HIGH_IMPRESSIONS_LOW_RANK,Optimize Immediately
6929507,2026-03-22,client_23a62021009f63c4,content_13ef8874a9a1ef5e,True,True,True,False,1,0,497,...,0.0,0.0,0.0,0.0,0.0,0.000025,0.997992,0.499008,HIGH_IMPRESSIONS_LOW_RANK,Optimize Immediately
8386759,2026-03-30,client_23a62021009f63c4,content_aacb637aa920b8ea,True,True,True,False,1,0,495,...,0.0,0.0,0.0,0.0,0.0,0.000025,0.993976,0.497000,HIGH_IMPRESSIONS_LOW_RANK,Optimize Immediately
8054586,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,True,True,True,True,39305,252,86373,...,0.0,0.0,0.0,0.0,26.0,0.980566,0.004413,0.492489,GOOD_POSITION,Protect Performance
103612,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,True,False,True,None,39003,2,107840,...,NaN,NaN,NaN,NaN,NaN,0.973032,0.005552,0.489292,GOOD_POSITION,Protect Performance
7005976,2026-03-23,client_23a62021009f63c4,content_c7ebdf81f488f0d8,True,True,True,False,1,0,480,...,0.0,0.0,0.0,0.0,0.0,0.000025,0.963855,0.481940,HIGH_IMPRESSIONS_LOW_RANK,Optimize Immediately
9179913,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,True,True,True,True,38436,271,84405,...,0.0,0.0,0.0,0.0,16.0,0.958886,0.004410,0.481648,GOOD_POSITION,Protect Performance
103661,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,True,False,True,None,37368,0,321886,...,NaN,NaN,NaN,NaN,NaN,0.932242,0.017297,0.474770,HIGH_IMPRESSIONS_LOW_RANK,Optimize Immediately
827924,2026-03-03,client_20259bd6705d81d4,content_8c34799f566ce23c,True,True,True,False,1,0,469,...,0.0,0.0,0.0,0.0,0.0,0.000025,0.941767,0.470896,HIGH_IMPRESSIONS_LOW_RANK,Optimize Immediately


# **Save Ranked Queue**

In [8]:
from pathlib import Path

output_dir = Path("submission")

output_dir.mkdir(exist_ok=True)

baseline.to_csv(
    output_dir / "baseline_ranked_queue.csv",
    index=False
)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# Select the top 20 ranked pages
top20 = baseline.head(20)
top20_review = top20[
[
    "content_hash_id",
    "gsc_impressions",
    "gsc_avg_position",
    "baseline_score",
    "reason_code",
    "action"
]
].copy()

In [10]:
top20_review["confidence_note"] = (

    "Medium confidence. The baseline uses only "
    "search impressions and average position."

)

top20_review["what_would_make_it_wrong"] = (

    "Temporary ranking fluctuations, seasonality, "
    "content quality, or missing business context."

)
top20_review

,content_hash_id,gsc_impressions,gsc_avg_position,baseline_score,reason_code,action,confidence_note,what_would_make_it_wrong
9655081,content_44f34c0a90047651,40084,0.083350,0.500084,GOOD_POSITION,Protect Performance,Medium confidence. The baseline uses only sear...,"Temporary ranking fluctuations, seasonality, c..."
3780826,content_8390ee56e6ea98ee,1,498.000000,0.500012,HIGH_IMPRESSIONS_LOW_RANK,Optimize Immediately,Medium confidence. The baseline uses only sear...,"Temporary ranking fluctuations, seasonality, c..."
6929507,content_13ef8874a9a1ef5e,1,497.000000,0.499008,HIGH_IMPRESSIONS_LOW_RANK,Optimize Immediately,Medium confidence. The baseline uses only sear...,"Temporary ranking fluctuations, seasonality, c..."
8386759,content_aacb637aa920b8ea,1,495.000000,0.497000,HIGH_IMPRESSIONS_LOW_RANK,Optimize Immediately,Medium confidence. The baseline uses only sear...,"Temporary ranking fluctuations, seasonality, c..."
8054586,content_eadb33b5df496f4a,39305,2.197507,0.492489,GOOD_POSITION,Protect Performance,Medium confidence. The baseline uses only sear...,"Temporary ranking fluctuations, seasonality, c..."
103612,content_34a70fea29d15f24,39003,2.764916,0.489292,GOOD_POSITION,Protect Performance,Medium confidence. The baseline uses only sear...,"Temporary ranking fluctuations, seasonality, c..."
7005976,content_c7ebdf81f488f0d8,1,480.000000,0.481940,HIGH_IMPRESSIONS_LOW_RANK,Optimize Immediately,Medium confidence. The baseline uses only sear...,"Temporary ranking fluctuations, seasonality, c..."
9179913,content_eadb33b5df496f4a,38436,2.195988,0.481648,GOOD_POSITION,Protect Performance,Medium confidence. The baseline uses only sear...,"Temporary ranking fluctuations, seasonality, c..."
103661,content_945d6ff91386c817,37368,8.613948,0.474770,HIGH_IMPRESSIONS_LOW_RANK,Optimize Immediately,Medium confidence. The baseline uses only sear...,"Temporary ranking fluctuations, seasonality, c..."
827924,content_8c34799f566ce23c,1,469.000000,0.470896,HIGH_IMPRESSIONS_LOW_RANK,Optimize Immediately,Medium confidence. The baseline uses only sear...,"Temporary ranking fluctuations, seasonality, c..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

The baseline considers only search impressions and average search position. It may incorrectly prioritize pages affected by temporary ranking fluctuations or pages with limited business value. This baseline serves as a simple benchmark that will later be compared with a machine learning ranking model using a richer feature set.

In [11]:
# Features used in the baseline rule
used_features = [
    "gsc_impressions",
    "gsc_avg_position"
]

print("Features used in the baseline:")
for feature in used_features:
    print("-", feature)

Features used in the baseline:
- gsc_impressions
- gsc_avg_position


In [12]:
# Look for columns that could indicate future windows
future_columns = [
    col for col in df.columns
    if "future" in col.lower()
    or "next" in col.lower()
    or "label" in col.lower()
]

print("Potential future/label columns:")
print(future_columns if future_columns else "None found")

Potential future/label columns:
None found


In [13]:
identifier_columns = [

    "report_date",

    "client_hash_id",

    "content_hash_id"

]

assert (

    len(

        set(identifier_columns)

        &

        set(["gsc_impressions","gsc_avg_position"])

    )

)==0

In [14]:
# Look for product-related columns
product_columns = [
    col for col in df.columns
    if "product" in col.lower() or "flag" in col.lower()
]

print("Potential product flag columns:")
print(product_columns if product_columns else "None found")

Potential product flag columns:
None found


In [15]:
print("Columns used to calculate baseline_score:")
print(used_features)

assert set(used_features) == {"gsc_impressions", "gsc_avg_position"}

print("Leakage check passed: baseline score uses only current ranking signals.")

Columns used to calculate baseline_score:
['gsc_impressions', 'gsc_avg_position']
Leakage check passed: baseline score uses only current ranking signals.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.